# 12.18 - Hybrid Search RAG

**Phase:** 12 - LangChain

**Status:** VERIFIED

---

## 1. What Are We Solving?

Semantic search misses exact keywords. Keyword search misses meaning. Hybrid search combines both.

## 2. Why Does This Matter?

Hybrid search gives the best of both worlds: semantic understanding + keyword precision.

## 3. Prerequisites

- 12.12: Embeddings & vector stores

## 4. Learning Objectives

- Implement keyword search (BM25)
- Implement semantic search (TF-IDF)
- Combine results with weighted fusion
- Compare retrieval strategies

## 5. Mental Model

Hybrid = alpha * semantic + (1-alpha) * keyword.
Alpha controls the blend.

In [1]:
import numpy as np
from langchain_core.documents import Document
from collections import Counter
import math
print("Libraries loaded.")

Libraries loaded.


## 6. Sample Documents

In [2]:
docs = [
    Document(page_content="Python is a high-level programming language.", metadata={"id": "1"}),
    Document(page_content="Machine learning algorithms learn from data.", metadata={"id": "2"}),
    Document(page_content="Deep learning uses neural networks with multiple layers.", metadata={"id": "3"}),
    Document(page_content="Natural language processing analyzes text and speech.", metadata={"id": "4"}),
    Document(page_content="Computer vision processes images and video.", metadata={"id": "5"}),
    Document(page_content="Reinforcement learning trains agents through rewards.", metadata={"id": "6"}),
]
print("Loaded " + str(len(docs)) + " documents")

Loaded 6 documents


## 7. Keyword Search (BM25)

In [3]:
def bm25_search(query, docs, k=3):
    """Simple BM25 implementation."""
    query_words = query.lower().split()
    doc_words = [d.page_content.lower().split() for d in docs]
    
    n = len(docs)
    df = Counter()
    for words in doc_words:
        for w in set(words):
            df[w] += 1
    
    scores = []
    for i, words in enumerate(doc_words):
        score = 0
        for q in query_words:
            if q in df:
                idf = math.log((n - df[q] + 0.5) / (df[q] + 0.5) + 1)
                tf = words.count(q) / len(words)
                score += idf * tf
        scores.append((docs[i], score))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:k]

results = bm25_search("neural networks", docs)
print("BM25 results:")
for doc, score in results:
    print("  Score: " + str(round(score, 4)) + " - " + doc.page_content[:50])

BM25 results:
  Score: 0.3851 - Deep learning uses neural networks with multiple l
  Score: 0.0 - Python is a high-level programming language.
  Score: 0.0 - Machine learning algorithms learn from data.


## 8. Semantic Search (TF-IDF)

In [4]:
def tfidf_embed(texts):
    """Compute TF-IDF vectors."""
    tokenized = [text.lower().split() for text in texts]
    vocab = set()
    for tokens in tokenized:
        vocab.update(tokens)
    vocab = sorted(vocab)
    word_to_idx = {w: i for i, w in enumerate(vocab)}
    
    n = len(texts)
    df = Counter()
    for tokens in tokenized:
        for w in set(tokens):
            df[w] += 1
    
    idf = {}
    for word in vocab:
        idf[word] = math.log((n + 1) / (df.get(word, 0) + 1)) + 1
    
    vectors = []
    for tokens in tokenized:
        tf = Counter(tokens)
        total = len(tokens)
        vec = np.zeros(len(vocab))
        for word, count in tf.items():
            if word in word_to_idx:
                vec[word_to_idx[word]] = (count / total) * idf[word]
        vectors.append(vec)
    
    return np.array(vectors), vocab, word_to_idx, idf

texts = [d.page_content for d in docs]
vectors, vocab, word_to_idx, idf = tfidf_embed(texts)
print("Vocab size:", len(vocab))
print("Vector shape:", vectors.shape)

Vocab size: 36
Vector shape: (6, 36)


In [5]:
def cosine_similarity(a, b):
    dot = np.dot(a, b)
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    return float(dot / norm) if norm > 0 else 0.0

def tfidf_search(query, docs, vectors, vocab, word_to_idx, idf, k=3):
    """Search using TF-IDF cosine similarity."""
    tokens = query.lower().split()
    tf = Counter(tokens)
    total = len(tokens)
    
    query_vec = np.zeros(len(vocab))
    for word, count in tf.items():
        if word in word_to_idx:
            query_vec[word_to_idx[word]] = (count / total) * idf.get(word, 1)
    
    scores = []
    for i, vec in enumerate(vectors):
        score = cosine_similarity(query_vec, vec)
        scores.append((docs[i], score))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:k]

results = tfidf_search("neural networks", docs, vectors, vocab, word_to_idx, idf)
print("TF-IDF results:")
for doc, score in results:
    print("  Score: " + str(round(score, 4)) + " - " + doc.page_content[:50])

TF-IDF results:


  Score: 0.5171 - Deep learning uses neural networks with multiple l
  Score: 0.0 - Python is a high-level programming language.
  Score: 0.0 - Machine learning algorithms learn from data.


## 9. Hybrid Search

In [6]:
def hybrid_search(query, docs, vectors, vocab, word_to_idx, idf, alpha=0.5, k=3):
    """Combine BM25 and TF-IDF search."""
    bm25_results = bm25_search(query, docs, k=k*2)
    tfidf_results = tfidf_search(query, docs, vectors, vocab, word_to_idx, idf, k=k*2)
    
    bm25_max = max(s for _, s in bm25_results) if bm25_results else 1
    tfidf_max = max(s for _, s in tfidf_results) if tfidf_results else 1
    
    combined = {}
    for doc, score in bm25_results:
        doc_id = doc.metadata["id"]
        combined[doc_id] = {"doc": doc, "score": alpha * (score / bm25_max)}
    
    for doc, score in tfidf_results:
        doc_id = doc.metadata["id"]
        if doc_id in combined:
            combined[doc_id]["score"] += (1 - alpha) * (score / tfidf_max)
        else:
            combined[doc_id] = {"doc": doc, "score": (1 - alpha) * (score / tfidf_max)}
    
    results = sorted(combined.values(), key=lambda x: x["score"], reverse=True)
    return [(r["doc"], r["score"]) for r in results[:k]]

results = hybrid_search("neural networks deep learning", docs, vectors, vocab, word_to_idx, idf, alpha=0.5)
print("Hybrid results:")
for doc, score in results:
    print("  Score: " + str(round(score, 4)) + " - " + doc.page_content[:50])

Hybrid results:
  Score: 1.0 - Deep learning uses neural networks with multiple l
  Score: 0.1674 - Machine learning algorithms learn from data.
  Score: 0.1674 - Reinforcement learning trains agents through rewar


## 10. Compare Strategies

In [7]:
query = "natural language processing"

print("Query:", query)
print("")
print("BM25 (top 2):")
for doc, score in bm25_search(query, docs, k=2):
    print("  -", doc.page_content[:50])

print("TF-IDF (top 2):")
for doc, score in tfidf_search(query, docs, vectors, vocab, word_to_idx, idf, k=2):
    print("  -", doc.page_content[:50])

print("Hybrid (top 2):")
for doc, score in hybrid_search(query, docs, vectors, vocab, word_to_idx, idf, alpha=0.5, k=2):
    print("  -", doc.page_content[:50])

Query: natural language processing

BM25 (top 2):
  - Natural language processing analyzes text and spee
  - Python is a high-level programming language.
TF-IDF (top 2):
  - Natural language processing analyzes text and spee
  - Python is a high-level programming language.
Hybrid (top 2):
  - Natural language processing analyzes text and spee
  - Python is a high-level programming language.


## 11. Common Mistakes

1. Wrong alpha value
2. Not normalizing scores
3. Ignoring one modality
4. Too small k for fusion

## 12. Coding Exercises

### Exercise 1: Tune Alpha
Find the best alpha for your data.

### Exercise 2: Compare Strategies
Evaluate all three strategies on test queries.

In [8]:
# EXERCISE 1
print("Exercise: Tune alpha parameter.")

Exercise: Tune alpha parameter.


In [9]:
# EXERCISE 2
print("Exercise: Compare retrieval strategies.")

Exercise: Compare retrieval strategies.


## 13. Closed-Book Recall

1. What does BM25 measure?
2. What does TF-IDF measure?
3. How do you combine them?

## 14. Summary

Hybrid search combines BM25 (keyword) and TF-IDF (semantic). Alpha controls the blend. Always normalize scores before combining.

## Verification Status
```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: [numpy, langchain-core]
OUTPUTS: PASS
LAST VERIFIED: 2026-08-30
```